> **Notebook-first lesson.** Run cells top-to-bottom. Environment-specific operations are written to be inspectable even when a service/device is unavailable.

## Mathematical Framework

Math companions for this lesson:

- [Math 04 · Statistics & Likelihood](../../math/04_statistics_likelihood.ipynb)
- [Math 06 · Optimization](../../math/06_optimization.ipynb)
- [Math 08 · Regularization & Generalization](../../math/08_regularization_generalization.ipynb)

Use the math to separate **measured association from assumptions, optimization behavior from generalization, and confidence scores from calibrated uncertainty**.

# Lesson 61: GPU performance engineering

## Goal
Learn why some training jobs underuse expensive hardware.

## Bottlenecks
- small batch sizes
- slow data loading
- CPU preprocessing
- host-device transfer
- synchronization
- memory pressure
- inefficient tensor shapes
- excessive Python overhead

## Tools/concepts
- pinned memory
- DataLoader workers
- mixed precision
- gradient accumulation
- profiling
- memory measurement

## Mixed precision


In [ ]:
from pathlib import Path
import os, sys
ROOT = Path(os.environ.get('COURSE_ROOT', next((str(p) for p in [Path.cwd(), *Path.cwd().parents] if (p / 'coursekit').is_dir()), '.'))).resolve()
if not (ROOT / 'coursekit').is_dir():
    raise RuntimeError('Open this notebook in the cloned ai-ml-learning repository.')
sys.path.insert(0, str(ROOT))
from coursekit.runtime import configure
SEED = 0
configure(SEED)
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
torch.manual_seed(SEED)
X_train = torch.randn(160, 2)
y_train = (X_train[:, 0] + .5 * X_train[:, 1] > 0).long()
X_val = torch.randn(60, 2)
y_val = (X_val[:, 0] + .5 * X_val[:, 1] > 0).long()
device = torch.device('cpu')
model = nn.Sequential(nn.Linear(2, 16), nn.ReLU(), nn.Linear(16, 2)).to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=.01)
loader = DataLoader(TensorDataset(X_train, y_train), batch_size=32, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val, y_val), batch_size=32)
xb, yb = next(iter(loader))


In [ ]:
with torch.autocast(device_type=xb.device.type, dtype=torch.bfloat16, enabled=xb.device.type == "cuda"):
    logits = model(xb)
    loss = loss_fn(logits, yb)



Use the appropriate scaler/workflow for your PyTorch version and hardware.

## Exercise
Profile a training loop before and after changing batch size, DataLoader workers and mixed precision.

## Rule
Optimize measured bottlenecks, not imagined ones.


## Runnable activity
Run this experiment and change at least one data, threshold, deployment, or systems assumption.

In [ ]:
import time, torch
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device",device)
for batch in [32,256,2048]:
    x=torch.randn(batch,1024,device=device)
    W=torch.randn(1024,512,device=device)
    if device.type=="cuda": torch.cuda.synchronize()
    t=time.perf_counter()
    for _ in range(20): y=x@W
    if device.type=="cuda": torch.cuda.synchronize()
    print("batch",batch,"20 matmuls sec",time.perf_counter()-t)
print("Measure before optimizing. Small workloads may not benefit from GPU transfer/launch overhead.")

## Engineering checkpoint
Record the metric/result, the assumption you changed, and what would make this experiment invalid in a real deployment.